# Parquet → DuckDB (Star Schema) — Colab

Lee los archivos **Parquet** generados por `csv_to_parquet_rapids_v2.ipynb` desde Google Drive
y construye un **Star Schema** en DuckDB, guardando la base de datos en una subcarpeta de Drive.

```
Drive/mortgage-risk/parquet/**/year=*/*.parquet
         ↓
Drive/mortgage-risk/duckdb/loans.duckdb
```

**Estrategia:**
- DuckDB lee los Parquet nativamente (`read_parquet` con Hive partitioning) — sin cargar todo en RAM
- Dims de enumeración (Channel, Purpose, Delinquency, etc.) → hardcodeadas con los valores del glosario
- Dims grandes (Loan, Borrower, Property, Seller, Servicer) → `SELECT DISTINCT` del Parquet con snapshot de originación
- `Dim_Date` → generada de todos los `Monthly_Reporting_Period` únicos del dataset
- `Fact_Loan_Performance` → un registro por `(Loan_Identifier, Monthly_Reporting_Period)`, FKs a todas las dims
- SCD Tipo 1 (sobrescritura) — suficiente para este ejercicio académico

## Recursos objetivo (Colab High-RAM)
- **RAM:** 47 GB
- **Disco:** 225 GB

## Configuración anti-OOM
- DuckDB: `memory_limit=46GB`, `threads=2`, `preserve_insertion_order=false`
- Temp en disco: `temp_directory=/content` + `max_temp_directory_size=200GB` (spill cuando RAM se llena)
- Lectura lazy: Parquet se lee nativamente sin cargar todo en RAM

## Configuración requerida
1. **Runtime → Change runtime type → CPU** (no necesita GPU)
2. Asegurate de que `csv_to_parquet_rapids_v2.ipynb` ya corrió y los Parquet están en Drive

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive montado en /content/drive")

Mounted at /content/drive
Drive montado en /content/drive


## 2. Instalar DuckDB

In [ ]:
!pip install -q duckdb pyarrow
import duckdb
print(f"DuckDB version: {duckdb.__version__}")

DuckDB version: 1.3.2


## 3. Parámetros (rutas en Google Drive)

Ajustá `DRIVE_BASE` y las subcarpetas según tu estructura en Drive.

- `DIR_PARQUET`: donde están los Parquet generados por `csv_to_parquet_rapids_v2.ipynb`
- `DIR_DUCKDB`: subcarpeta donde se guardará el archivo `.duckdb`

In [ ]:
import os, glob

DRIVE_BASE  = "/content/drive/MyDrive"
DIR_PARQUET = "mortgage-risk/parquet"
DIR_DUCKDB  = "mortgage-risk/duckdb"

PARQUET_BASE = f"{DRIVE_BASE}/{DIR_PARQUET}"
DUCKDB_DIR   = f"{DRIVE_BASE}/{DIR_DUCKDB}"
DUCKDB_PATH  = f"{DUCKDB_DIR}/loans.duckdb"

# Glob recursivo para leer todos los parquet de todos los CSVs y años
# Estructura: PARQUET_BASE/<csv_stem>/year=YYYY/part-NNNN.snappy.parquet
PARQUET_GLOB = f"{PARQUET_BASE}/**/*.parquet"

# Crear carpeta de destino para DuckDB
os.makedirs(DUCKDB_DIR, exist_ok=True)

# Contar archivos disponibles
parquet_files = glob.glob(PARQUET_GLOB, recursive=True)

print(f"Rutas configuradas:")
print(f"  Parquet (entrada): {PARQUET_BASE}")
print(f"  DuckDB  (salida):  {DUCKDB_PATH}")
print(f"  Archivos Parquet encontrados: {len(parquet_files)}")

if not parquet_files:
    raise FileNotFoundError(
        "No se encontraron archivos Parquet. Ejecutá primero csv_to_parquet_rapids_v2.ipynb "
        "y asegurate de que los Parquet estén en Drive."
    )

# Extraer años únicos para procesamiento por partición (anti-OOM)
import re
YEARS = sorted(set(m.group(1) for f in parquet_files for m in [re.search(r'year=(\d+)', f)] if m))
print(f"  Años detectados: {len(YEARS)} ({YEARS[0]}-{YEARS[-1] if YEARS else '?'})")

# Detectar primera ejecución (DuckDB aún no existe)
IS_FIRST_RUN = not os.path.exists(DUCKDB_PATH)
if IS_FIRST_RUN:
    print("\n🆕 Primera ejecución: se creará loans.duckdb desde cero.")
else:
    print("\n📂 DuckDB existente: se reconstruirá el Star Schema (CREATE OR REPLACE).")

Rutas configuradas:
  Parquet (entrada): /content/drive/MyDrive/mortgage-risk/parquet
  DuckDB  (salida):  /content/drive/MyDrive/mortgage-risk/duckdb/loans.duckdb
  Archivos Parquet encontrados: 6564

📂 DuckDB existente: se reconstruirá el Star Schema (CREATE OR REPLACE).


## 4. Conectar a DuckDB y crear vista de staging

DuckDB lee los Parquet con Hive partitioning (`year=YYYY`) **sin cargar todo en RAM**.

Config: 46 GB RAM, temp en `/content` (200 GB spill), 2 threads, `preserve_insertion_order=false`.

In [ ]:
import duckdb

# Conectar a DuckDB (crea el archivo automáticamente si no existe)
con = duckdb.connect(DUCKDB_PATH)

# Ajustes para 47 GB RAM + 225 GB disco (anti-OOM)
os.makedirs("/content/duckdb_temp", exist_ok=True)
con.execute("SET temp_directory = '/content/duckdb_temp'")
con.execute("PRAGMA max_temp_directory_size = '200GB'")
con.execute("SET memory_limit = '46GB'")
con.execute("SET threads TO 2")
con.execute("SET preserve_insertion_order = false")

# Vista de staging sobre todos los Parquet
# hive_partitioning detecta automáticamente year=YYYY en las rutas
con.execute(f"""
    CREATE OR REPLACE VIEW stg_loans AS
    SELECT * FROM read_parquet('{PARQUET_GLOB}', hive_partitioning := true)
""")

n    = con.execute("SELECT COUNT(*) FROM stg_loans").fetchone()[0]
cols = con.execute("SELECT COUNT(*) FROM pragma_table_info('stg_loans')").fetchone()[0]

print(f"✓ Vista stg_loans: {n:,} filas × {cols} columnas")
print(f"✓ DuckDB conectado: {DUCKDB_PATH}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Vista stg_loans: 3,108,993,566 filas × 111 columnas
✓ DuckDB conectado: /content/drive/MyDrive/mortgage-risk/duckdb/loans.duckdb


## 5. Dimensiones de enumeración (hardcoded)

Tablas pequeñas cuyos valores provienen directamente del glosario de Freddie Mac.

In [ ]:
# ── Dim_Channel ──────────────────────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE Dim_Channel (
        Channel_Key  INTEGER PRIMARY KEY,
        Channel_Code VARCHAR(1),
        Channel_Desc VARCHAR(50)
    )
""")
con.executemany("INSERT INTO Dim_Channel VALUES (?, ?, ?)", [
    (0, None,  'Unknown'),
    (1, 'R',   'Retail'),
    (2, 'C',   'Correspondent'),
    (3, 'B',   'Broker'),
])

# ── Dim_Loan_Purpose ─────────────────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE Dim_Loan_Purpose (
        Purpose_Key  INTEGER PRIMARY KEY,
        Purpose_Code VARCHAR(1),
        Purpose_Desc VARCHAR(60)
    )
""")
con.executemany("INSERT INTO Dim_Loan_Purpose VALUES (?, ?, ?)", [
    (0, None, 'Unknown'),
    (1, 'P',  'Purchase'),
    (2, 'C',  'Cash-Out Refinance'),
    (3, 'R',  'Rate/Term Refinance'),
    (4, 'U',  'Not Specified'),
])

# ── Dim_Amortization_Type ────────────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE Dim_Amortization_Type (
        Amortization_Key  INTEGER PRIMARY KEY,
        Amortization_Code VARCHAR(3),
        Amortization_Desc VARCHAR(50)
    )
""")
con.executemany("INSERT INTO Dim_Amortization_Type VALUES (?, ?, ?)", [
    (0, None,  'Unknown'),
    (1, 'FRM', 'Fixed-Rate Mortgage'),
    (2, 'ARM', 'Adjustable-Rate Mortgage'),
])

# ── Dim_Occupancy_Status ─────────────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE Dim_Occupancy_Status (
        Occupancy_Key  INTEGER PRIMARY KEY,
        Occupancy_Code VARCHAR(1),
        Occupancy_Desc VARCHAR(30)
    )
""")
con.executemany("INSERT INTO Dim_Occupancy_Status VALUES (?, ?, ?)", [
    (0, None, 'Unknown'),
    (1, 'P',  'Primary Residence'),
    (2, 'S',  'Second Home'),
    (3, 'I',  'Investment Property'),
    (4, 'U',  'Not Specified'),
])

# ── Dim_Property_Type ────────────────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE Dim_Property_Type (
        Property_Type_Key  INTEGER PRIMARY KEY,
        Property_Type_Code VARCHAR(2),
        Property_Type_Desc VARCHAR(50)
    )
""")
con.executemany("INSERT INTO Dim_Property_Type VALUES (?, ?, ?)", [
    (0, None, 'Unknown'),
    (1, 'SF', 'Single Family'),
    (2, 'CO', 'Condominium'),
    (3, 'CP', 'Co-operative'),
    (4, 'PU', 'Planned Unit Development'),
    (5, 'MH', 'Manufactured Housing'),
])

# ── Dim_Delinquency_Status ───────────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE Dim_Delinquency_Status (
        Delinquency_Key  INTEGER PRIMARY KEY,
        Status_Code      VARCHAR(2),
        Status_Desc      VARCHAR(60),
        Days_Delinquent  INTEGER
    )
""")
con.executemany("INSERT INTO Dim_Delinquency_Status VALUES (?, ?, ?, ?)", [
    (0,  None, 'Unknown',                       None),
    (1,  '00', 'Current',                           0),
    (2,  '01', '30-59 Days Past Due',              30),
    (3,  '02', '60-89 Days Past Due',              60),
    (4,  '03', '90-119 Days Past Due',             90),
    (5,  '04', '120-149 Days Past Due',           120),
    (6,  '05', '150-179 Days Past Due',           150),
    (7,  '06', '180+ Days Past Due',              180),
    (8,  'XX', 'Foreclosure / REO',              None),
    (9,  '99', 'Unavailable / Not Reported',     None),
])

# ── Dim_Zero_Balance_Reason ──────────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE Dim_Zero_Balance_Reason (
        Zero_Balance_Key  INTEGER PRIMARY KEY,
        Code              VARCHAR(2),
        Reason_Desc       VARCHAR(80)
    )
""")
con.executemany("INSERT INTO Dim_Zero_Balance_Reason VALUES (?, ?, ?)", [
    (0,  None, 'Active / Not Resolved'),
    (1,  '01', 'Prepaid or Matured'),
    (2,  '02', 'Third Party Sale'),
    (3,  '03', 'Short Sale / Deed-in-Lieu'),
    (4,  '06', 'Repurchase Prior to Sale'),
    (5,  '09', 'REO Disposition'),
    (6,  '15', 'Reperforming Loan Sale'),
    (7,  '16', 'Reperforming Loan Repurchase'),
    (8,  '96', 'Removal – Reclassification ARM→Fixed'),
    (9,  '97', 'Removal – Reclassification Fixed→ARM'),
    (10, '98', 'Other Removal'),
])

print("✓ Dimensiones de enumeración creadas")
for tbl in ['Dim_Channel','Dim_Loan_Purpose','Dim_Amortization_Type',
            'Dim_Occupancy_Status','Dim_Property_Type',
            'Dim_Delinquency_Status','Dim_Zero_Balance_Reason']:
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<35} {n} filas")

✓ Dimensiones de enumeración creadas
  Dim_Channel                         4 filas
  Dim_Loan_Purpose                    5 filas
  Dim_Amortization_Type               3 filas
  Dim_Occupancy_Status                5 filas
  Dim_Property_Type                   6 filas
  Dim_Delinquency_Status              10 filas
  Dim_Zero_Balance_Reason             11 filas


## 6. Dimensiones grandes (desde los Parquet)

DuckDB hace `SELECT DISTINCT` directamente sobre la vista `stg_loans` sin cargar todo en memoria.

In [ ]:
import time
t0 = time.time()

# ── Dim_Date (desde Monthly_Reporting_Period) ─────────────────────────────────
# Date_Key = INTEGER YYYYMM (e.g. 200001 para Enero 2000)
print("Creando Dim_Date...")
con.execute("""
    CREATE OR REPLACE TABLE Dim_Date AS
    SELECT DISTINCT
        CAST(YEAR(Monthly_Reporting_Period) * 100 + MONTH(Monthly_Reporting_Period) AS INTEGER) AS Date_Key,
        YEAR(Monthly_Reporting_Period)      AS Year,
        QUARTER(Monthly_Reporting_Period)   AS Quarter,
        MONTH(Monthly_Reporting_Period)     AS Month,
        STRFTIME(Monthly_Reporting_Period, '%Y-%m')  AS Year_Month,
        MONTHNAME(Monthly_Reporting_Period)           AS Month_Name,
        (YEAR(Monthly_Reporting_Period) % 4 = 0
            AND (YEAR(Monthly_Reporting_Period) % 100 != 0
              OR YEAR(Monthly_Reporting_Period) % 400 = 0)) AS Is_Leap_Year,
        Monthly_Reporting_Period AS Date_Value
    FROM stg_loans
    WHERE Monthly_Reporting_Period IS NOT NULL
    ORDER BY Date_Key
""")

# ── Dim_Seller ────────────────────────────────────────────────────────────────
print("Creando Dim_Seller...")
con.execute("""
    CREATE OR REPLACE TABLE Dim_Seller AS
    SELECT
        ROW_NUMBER() OVER () AS Seller_Key,
        COALESCE(Seller_Name, 'Unknown') AS Seller_Name
    FROM (
        SELECT DISTINCT Seller_Name FROM stg_loans
    )
""")

# ── Dim_Servicer ──────────────────────────────────────────────────────────────
print("Creando Dim_Servicer...")
con.execute("""
    CREATE OR REPLACE TABLE Dim_Servicer AS
    SELECT
        ROW_NUMBER() OVER () AS Servicer_Key,
        COALESCE(Servicer_Name, 'Unknown') AS Servicer_Name
    FROM (
        SELECT DISTINCT Servicer_Name FROM stg_loans
    )
""")

# ── Dim_Loan, Dim_Borrower, Dim_Property: PROCESAMIENTO POR AÑO (anti-OOM) ─────
# Cada año se procesa por separado; luego se fusiona manteniendo min(Loan_Age) por préstamo
import gc

for dim_name, select_cols in [
    ("Dim_Loan", "Loan_Identifier, Origination_Date, First_Payment_Date, Maturity_Date, Original_Loan_Term, Original_Interest_Rate, Original_Loan_to_Value_Ratio_LTV AS Original_LTV, Original_Combined_Loan_to_Value_Ratio_CLTV AS Original_CLTV, \"Debt-To-Income_DTI\" AS DTI_at_Origination, Mortgage_Insurance_Percentage, Prepayment_Penalty_Indicator, Interest_Only_Loan_Indicator, High_Balance_Loan_Indicator, \"High_Loan_to_Value_HLTV_Refinance_Option_Indicator\" AS HLTV_Refi_Indicator, Relocation_Mortgage_Indicator, Amortization_Type, Reference_Pool_ID, Deal_Name, Loan_Age"),
    ("Dim_Borrower", "Loan_Identifier, Number_of_Borrowers, Borrower_Credit_Score_at_Origination, \"Co-Borrower_Credit_Score_at_Origination\", First_Time_Home_Buyer_Indicator, Loan_Age"),
    ("Dim_Property", "Loan_Identifier, Property_Type, Number_of_Units, Occupancy_Status, Property_State, Metropolitan_Statistical_Area_MSA AS MSA_Code, Zip_Code_Short, Property_Valuation_Method, Loan_Age"),
]:
    print(f"Creando {dim_name} (por año)...")
    con.execute(f"DROP TABLE IF EXISTS {dim_name}")
    for i, yr in enumerate(YEARS):
        glob_yr = f"{PARQUET_BASE}/**/year={yr}/*.parquet"
        con.execute(f"""
            CREATE TABLE stg_{dim_name}_{yr} AS
            WITH first_obs AS (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY Loan_Identifier ORDER BY COALESCE(Loan_Age, 999)) AS rn
                FROM read_parquet('{glob_yr}', hive_partitioning := true)
            )
            SELECT {select_cols} FROM first_obs WHERE rn = 1
        """)
        if i == 0:
            con.execute(f"CREATE TABLE {dim_name} AS SELECT * FROM stg_{dim_name}_{yr}")
        else:
            con.execute(f"""
                CREATE OR REPLACE TABLE {dim_name} AS
                SELECT * EXCLUDE (rn) FROM (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY Loan_Identifier ORDER BY COALESCE(Loan_Age, 999)) AS rn
                    FROM (SELECT * FROM {dim_name} UNION ALL SELECT * FROM stg_{dim_name}_{yr})
                ) WHERE rn = 1
            """)
        con.execute(f"DROP TABLE stg_{dim_name}_{yr}")
        gc.collect()
        if (i + 1) % 5 == 0:
            print(f"  {dim_name}: {i+1}/{len(YEARS)} años")
    key_col = dim_name.replace('Dim_','') + '_Key'
    con.execute(f"ALTER TABLE {dim_name} DROP COLUMN Loan_Age")
    con.execute(f"CREATE OR REPLACE TABLE {dim_name} AS SELECT ROW_NUMBER() OVER () AS {key_col}, * FROM {dim_name}")


elapsed = time.time() - t0
print(f"\n✓ Dimensiones grandes creadas en {elapsed:.1f}s")
for tbl in ['Dim_Date','Dim_Seller','Dim_Servicer','Dim_Loan','Dim_Borrower','Dim_Property']:
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<30} {n:,} filas")

Creando Dim_Date...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Creando Dim_Seller...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Creando Dim_Servicer...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Creando Dim_Loan...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: failed to offload data block of size 32.0 KiB (156.9 GiB/156.9 GiB used).
This limit was set by the 'max_temp_directory_size' setting.
By default, this setting utilizes the available disk space on the drive where the 'temp_directory' is located.
You can adjust this setting, by using (for example) PRAGMA max_temp_directory_size='10GiB'

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

## 7. Fact_Loan_Performance

Un registro por `(Loan_Identifier, Monthly_Reporting_Period)`.  
Las métricas que cambian mensualmente (current UPB, tasa actual, credit score actual, delinquency)
se almacenan como **medidas** en el Fact, no como FKs adicionales.

⚠️ Esta celda puede tardar varios minutos dependiendo del volumen de datos.

In [ ]:
import time
t0 = time.time()
print("Construyendo Fact_Loan_Performance (puede tardar)...")

con.execute("""
    CREATE OR REPLACE TABLE Fact_Loan_Performance AS
    SELECT
        -- Surrogate keys (FKs a dimensiones)
        dl.Loan_Key,
        CAST(YEAR(s.Monthly_Reporting_Period) * 100
             + MONTH(s.Monthly_Reporting_Period) AS INTEGER)  AS Date_Key,
        db.Borrower_Key,
        dp.Property_Key,
        COALESCE(dsl.Seller_Key,   0)   AS Seller_Key,
        COALESCE(dsv.Servicer_Key, 0)   AS Servicer_Key,
        COALESCE(dc.Channel_Key,        0) AS Channel_Key,
        COALESCE(dpur.Purpose_Key,      0) AS Purpose_Key,
        COALESCE(dam.Amortization_Key,  0) AS Amortization_Key,
        COALESCE(ddel.Delinquency_Key,  0) AS Delinquency_Key,
        COALESCE(dzb.Zero_Balance_Key,  0) AS Zero_Balance_Key,

        -- Claves de negocio útiles para debug
        s.Loan_Identifier,
        s.Monthly_Reporting_Period,

        -- Medidas — saldos y tasas
        s.Current_Actual_UPB,
        s.Original_UPB,
        s.UPB_at_Issuance,
        s.Interest_Bearing_UPB,
        s.Current_Interest_Rate,
        s.Loan_Age,
        s.Remaining_Months_to_Legal_Maturity,
        s.Remaining_Months_To_Maturity,

        -- Medidas — scores actuales (cambian mensualmente)
        s.Borrower_Credit_Score_Current,
        s."Co-Borrower_Credit_Score_Current",

        -- Medidas — principal
        s.Scheduled_Principal_Current,
        s.Total_Principal_Current,
        s.Unscheduled_Principal_Current,

        -- Medidas — pérdidas y modificaciones
        s.Modification_Related_Non_Interest_Bearing_UPB,
        s.Principal_Forgiveness_Amount,
        s.Total_Deferral_Amount,
        s.Current_Period_Modification_Loss_Amount,
        s.Cumulative_Modification_Loss_Amount,
        s.Current_Period_Credit_Event_Net_Gain_or_Loss,
        s.Cumulative_Credit_Event_Net_Gain_or_Loss,

        -- Medidas — foreclosure y REO
        s.Foreclosure_Costs,
        s.Property_Preservation_and_Repair_Costs,
        s.Net_Sales_Proceeds,
        s.Credit_Enhancement_Proceeds,
        s.Repurchase_Make_Whole_Proceeds,
        s.Foreclosure_Principal_Write_off_Amount,
        s.Delinquent_Accrued_Interest,

        -- Flags binarios convertidos a 0/1 para agregación
        CASE WHEN s.Modification_Flag = 'Y' THEN 1 ELSE 0 END AS Is_Modified,
        CASE WHEN s.Servicing_Activity_Indicator = 'Y' THEN 1 ELSE 0 END AS Is_Active_Servicing,

        -- Columna de partición heredada del Parquet (cohort del CSV original)
        s.year AS cohort_year

    FROM stg_loans s

    -- Dims grandes (por Loan_Identifier)
    LEFT JOIN Dim_Loan       dl  ON s.Loan_Identifier = dl.Loan_Identifier
    LEFT JOIN Dim_Borrower   db  ON s.Loan_Identifier = db.Loan_Identifier
    LEFT JOIN Dim_Property   dp  ON s.Loan_Identifier = dp.Loan_Identifier
    LEFT JOIN Dim_Seller     dsl ON s.Seller_Name     = dsl.Seller_Name
    LEFT JOIN Dim_Servicer   dsv ON s.Servicer_Name   = dsv.Servicer_Name

    -- Dims de enumeración (por código)
    LEFT JOIN Dim_Channel           dc   ON s.Channel                          = dc.Channel_Code
    LEFT JOIN Dim_Loan_Purpose      dpur ON s.Loan_Purpose                     = dpur.Purpose_Code
    LEFT JOIN Dim_Amortization_Type dam  ON s.Amortization_Type                = dam.Amortization_Code
    LEFT JOIN Dim_Delinquency_Status ddel ON s.Current_Loan_Delinquency_Status = ddel.Status_Code
    LEFT JOIN Dim_Zero_Balance_Reason dzb ON s.Zero_Balance_Code               = dzb.Code
""")

elapsed = time.time() - t0
n_fact = con.execute("SELECT COUNT(*) FROM Fact_Loan_Performance").fetchone()[0]
print(f"✓ Fact_Loan_Performance: {n_fact:,} filas  ({elapsed:.1f}s)")

## 8. Verificación del Star Schema

In [ ]:
import pandas as pd

print("=" * 70 + "\nTABLAS EN DUCKDB\n" + "=" * 70)
tables = con.execute("""
    SELECT table_name, estimated_size
    FROM duckdb_tables()
    ORDER BY table_name
""").df()
print(tables.to_string(index=False))

print("\n" + "=" * 70 + "\nQUERY DE PRUEBA: UPB medio por trimestre y canal\n" + "=" * 70)
q1 = con.execute("""
    SELECT
        dd.Year,
        dd.Quarter,
        dc.Channel_Desc,
        COUNT(*)                              AS n_obs,
        COUNT(DISTINCT f.Loan_Identifier)     AS n_loans,
        ROUND(AVG(f.Current_Actual_UPB), 2)   AS avg_upb,
        ROUND(AVG(f.Current_Interest_Rate), 4) AS avg_rate
    FROM Fact_Loan_Performance f
    JOIN Dim_Date    dd ON f.Date_Key    = dd.Date_Key
    JOIN Dim_Channel dc ON f.Channel_Key = dc.Channel_Key
    GROUP BY dd.Year, dd.Quarter, dc.Channel_Desc
    ORDER BY dd.Year, dd.Quarter, dc.Channel_Desc
    LIMIT 20
""").df()
print(q1.to_string(index=False))

print("\n" + "=" * 70 + "\nQUERY DE PRUEBA: Distribución de delinquency\n" + "=" * 70)
q2 = con.execute("""
    SELECT
        ds.Status_Desc,
        COUNT(*) AS n_obs,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM Fact_Loan_Performance f
    JOIN Dim_Delinquency_Status ds ON f.Delinquency_Key = ds.Delinquency_Key
    GROUP BY ds.Status_Desc
    ORDER BY n_obs DESC
""").df()
print(q2.to_string(index=False))

print("\n" + "=" * 70 + "\nQUERY DE PRUEBA: Préstamos por estado (top 10)\n" + "=" * 70)
q3 = con.execute("""
    SELECT
        dp.Property_State,
        COUNT(DISTINCT f.Loan_Identifier) AS n_loans,
        ROUND(AVG(dl.Original_LTV), 2)    AS avg_orig_ltv,
        ROUND(AVG(dl.DTI_at_Origination), 2) AS avg_dti
    FROM Fact_Loan_Performance f
    JOIN Dim_Property dp ON f.Property_Key = dp.Property_Key
    JOIN Dim_Loan     dl ON f.Loan_Key     = dl.Loan_Key
    GROUP BY dp.Property_State
    ORDER BY n_loans DESC
    LIMIT 10
""").df()
print(q3.to_string(index=False))

## 9. Cerrar conexión y verificar archivo en Drive

DuckDB guarda automáticamente los cambios al cerrar la conexión.
El archivo `.duckdb` quedará persistido en tu Google Drive.

In [ ]:
import os

con.close()
print("✓ Conexión DuckDB cerrada — datos persistidos en Drive")

# Verificar que el archivo existe y mostrar su tamaño
if os.path.exists(DUCKDB_PATH):
    size_mb = os.path.getsize(DUCKDB_PATH) / (1024 ** 2)
    print(f"\n✅ Archivo guardado exitosamente:")
    print(f"   Ruta:  {DUCKDB_PATH}")
    print(f"   Tamaño: {size_mb:.1f} MB")
else:
    print("❌ El archivo no se encontró — verificá que Drive esté montado correctamente")